# WGAN-EEG FPGA Inference — PYNQ PS Controller

Notebook ini menjalankan pipeline inferensi 4-layer transpose convolution di FPGA,
mengikuti alur yang sama dengan `System_Top_Level_tb.v`.

**Interface:**
- `s0_axis` (DMA0 send) → Weight stream
- `s1_axis` (DMA1 send) → IFMAP stream  
- `s2_axis` (DMA2 send) → Bias stream
- `m0_axis` (DMA0 recv) → Notifikasi batch + Output data (BRAM 0–7)
- `m1_axis` (DMA1 recv) → Output data (BRAM 8–15)

**Packet header format (6 × 24-bit words stored in uint32):**
```
Word 0: 0x00C0DE  (magic write)
Word 1: 0x000001  (instruction = write mode)
Word 2: 0         (bram_start = 0)
Word 3: 15        (bram_end   = 15)
Word 4: 0         (padding)
Word 5: N         (num_words_per_bram)
```

**Notification packet** dari FPGA ke PS (m0):
- 6 kata, header[0] bit[15:0] = `0xC0DE`, `tlast` di kata ke-6
- `header[2] bit[2:0]` = batch_id yang selesai

**Output data packet** dari FPGA ke PS (m0 + m1):
- m0: 6-word header (`header[0][15:0] = 0xDA7A`) + 4096 data words
- m1: 4096 data words (tanpa header, bersamaan dengan fase data m0)

In [ ]:
# ============================================================
# CELL 1: Imports & Konfigurasi
# ============================================================
import numpy as np
import threading
import time
import os

from pynq import Overlay, allocate

# ----- Path file .mem (sesuaikan dengan lokasi di PYNQ) -----
DATA_DIR = "/home/xilinx/wgan_eeg/"   # <-- ubah sesuai path aktual

WEIGHT_MEM_FILE = DATA_DIR + "G_d5_Q9.14_decoder_weight.mem"
BIAS_MEM_FILE   = DATA_DIR + "G_d5_Q9.14_decoder_bias.mem"
IFMAP_L0_FILE   = DATA_DIR + "b3.mem"
IFMAP_L1_FILE   = DATA_DIR + "d2_in.mem"
IFMAP_L2_FILE   = DATA_DIR + "d3_in.mem"
IFMAP_L3_FILE   = DATA_DIR + "d4_in.mem"

# ----- Offset dalam file .mem gabungan -----
WEIGHT_OFFSET_L0 = 0
WEIGHT_OFFSET_L1 = 131072
WEIGHT_OFFSET_L2 = 196608
WEIGHT_OFFSET_L3 = 212992

BIAS_OFFSET_L0 = 0
BIAS_OFFSET_L1 = 8192
BIAS_OFFSET_L2 = 16384
BIAS_OFFSET_L3 = 24576

print("Imports OK")

In [ ]:
# ============================================================
# CELL 2: Load Overlay dan Setup DMA
# ============================================================
# Ubah path .bit/.hwh sesuai nama file bitstream kamu
overlay = Overlay("/home/xilinx/wgan_eeg/wgan_eeg_fpga.bit")

# AXI DMA IPs — sesuaikan nama dengan Vivado block design
# dma_weight : s0_axis (send) + m0_axis (recv — notifikasi & output)
# dma_ifmap  : s1_axis (send) + m1_axis (recv — output)
# dma_bias   : s2_axis (send only)
dma_weight = overlay.axi_dma_0   # <-- sesuaikan nama
dma_ifmap  = overlay.axi_dma_1   # <-- sesuaikan nama
dma_bias   = overlay.axi_dma_2   # <-- sesuaikan nama

print("Overlay loaded")
print(f"  dma_weight: {dma_weight}")
print(f"  dma_ifmap : {dma_ifmap}")
print(f"  dma_bias  : {dma_bias}")

In [ ]:
# ============================================================
# CELL 3: Load data dari file .mem
# ============================================================
def load_mem_file(filepath):
    """Load Verilog $readmemh .mem file → numpy uint32 array."""
    values = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.split('//')[0].strip()
            if not line or line.startswith('@'):
                continue
            values.append(int(line, 16))
    return np.array(values, dtype=np.uint32)

print("Loading weight...", end=' ', flush=True)
weight_data = load_mem_file(WEIGHT_MEM_FILE)
print(f"{len(weight_data)} words")

print("Loading bias...", end=' ', flush=True)
bias_data = load_mem_file(BIAS_MEM_FILE)
print(f"{len(bias_data)} words")

print("Loading ifmap L0...", end=' ', flush=True)
ifmap_l0 = load_mem_file(IFMAP_L0_FILE)
print(f"{len(ifmap_l0)} words")

print("Loading ifmap L1...", end=' ', flush=True)
ifmap_l1 = load_mem_file(IFMAP_L1_FILE)
print(f"{len(ifmap_l1)} words")

print("Loading ifmap L2...", end=' ', flush=True)
ifmap_l2 = load_mem_file(IFMAP_L2_FILE)
print(f"{len(ifmap_l2)} words")

print("Loading ifmap L3...", end=' ', flush=True)
ifmap_l3 = load_mem_file(IFMAP_L3_FILE)
print(f"{len(ifmap_l3)} words")

print("Data loaded OK")

In [ ]:
# ============================================================
# CELL 4: Helper — Packet Builder
# ============================================================

def _header(num_words_per_bram):
    """6-word packet header."""
    return [0x00C0DE, 0x000001, 0, 15, 0, int(num_words_per_bram)]

def _pack(arr):
    """Convert Python list → numpy uint32, mask to 24-bit."""
    return np.array(arr, dtype=np.uint32) & 0xFFFFFF

# ---- WEIGHT packets ----

def build_weight_l0(batch_id, weight):
    """
    Layer 0 weight packet, 1 batch.
    Per batch: 16 BRAMs × 1024 words = 16384 words.
    OC base = batch_id × 16.
    DDR: OFFSET_L0 + (oc × 1024) + (k × 256) + ich
    """
    buf = _header(1024)
    for bram_id in range(16):
        k_pos      = bram_id & 0x3
        oc_in_tile = (bram_id >> 2) & 0x3
        for tile in range(4):
            oc_abs = batch_id * 16 + tile * 4 + oc_in_tile
            for ich in range(256):
                addr = WEIGHT_OFFSET_L0 + oc_abs * 1024 + k_pos * 256 + ich
                buf.append(int(weight[addr]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

def build_weight_l1(batch_id, weight):
    """
    Layer 1 weight packet, 1 batch.
    Per batch: 16 BRAMs × 1024 words = 16384 words.
    DDR: OFFSET_L1 + (oc × 1024) + (k × 256) + ich
    """
    buf = _header(1024)
    for bram_id in range(16):
        k_pos      = bram_id & 0x3
        oc_in_tile = (bram_id >> 2) & 0x3
        for tile in range(4):
            oc_abs = batch_id * 16 + tile * 4 + oc_in_tile
            for ich in range(256):
                addr = WEIGHT_OFFSET_L1 + oc_abs * 1024 + k_pos * 256 + ich
                buf.append(int(weight[addr]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

def build_weight_l2(weight):
    """
    Layer 2 weight packet, 1 batch saja.
    16 BRAMs × 1024 words, 8 tiles × 4 OC/tile × 128 IC.
    DDR: OFFSET_L2 + (oc × 512) + (k × 128) + ich
    """
    buf = _header(1024)
    for bram_id in range(16):
        k_pos      = bram_id & 0x3
        oc_in_tile = (bram_id >> 2) & 0x3
        for tile in range(8):
            oc_abs = tile * 4 + oc_in_tile
            for ich in range(128):
                addr = WEIGHT_OFFSET_L2 + oc_abs * 512 + k_pos * 128 + ich
                buf.append(int(weight[addr]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

def build_weight_l3(weight):
    """
    Layer 3 weight packet, 1 batch saja.
    16 BRAMs × 256 words, 4 tiles × 4 OC/tile × 64 IC.
    DDR: OFFSET_L3 + (oc × 256) + (k × 64) + cin
    """
    buf = _header(256)
    for bram_id in range(16):
        k_pos      = bram_id & 0x3
        oc_in_tile = (bram_id >> 2) & 0x3
        for tile in range(4):
            oc_abs = tile * 4 + oc_in_tile
            for cin in range(64):
                addr = WEIGHT_OFFSET_L3 + oc_abs * 256 + k_pos * 64 + cin
                buf.append(int(weight[addr]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

# ---- IFMAP packets ----

def build_ifmap_l0(ifmap):
    """
    Layer 0 IFMAP: 32pos × 256ch = 8192 words.
    Stride 16: BRAM_n menyimpan pos [n, n+16] × 256ch.
    DDR layout: pos × 256 + ch
    """
    buf = _header(512)   # 512 words per BRAM
    for bram_id in range(16):
        for pos_group in range(2):
            position = bram_id + pos_group * 16
            for ch in range(256):
                idx = position * 256 + ch
                buf.append(int(ifmap[idx]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

def build_ifmap_l1(ifmap):
    """
    Layer 1 IFMAP: 64pos × 256ch = 16384 words.
    Stride 16: BRAM_n menyimpan pos [n, n+16, n+32, n+48] × 256ch.
    """
    buf = _header(1024)   # 1024 words per BRAM
    for bram_id in range(16):
        for pos_group in range(4):
            position = bram_id + pos_group * 16
            for ch in range(256):
                idx = position * 256 + ch
                buf.append(int(ifmap[idx]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

def build_ifmap_l2(ifmap):
    """
    Layer 2 IFMAP: 128pos × 128ch = 16384 words.
    Stride 16: BRAM_n menyimpan pos [n, n+16, ..., n+112] × 128ch.
    """
    buf = _header(1024)
    for bram_id in range(16):
        for pos_group in range(8):
            position = bram_id + pos_group * 16
            for ch in range(128):
                idx = position * 128 + ch
                buf.append(int(ifmap[idx]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

def build_ifmap_l3(ifmap):
    """
    Layer 3 IFMAP: 256pos × 64ch = 16384 words.
    Stride 16: BRAM_n menyimpan pos [n, n+16, ..., n+240] × 64ch.
    """
    buf = _header(1024)
    for bram_id in range(16):
        for pos_group in range(16):
            position = bram_id + pos_group * 16
            for ch in range(64):
                idx = position * 64 + ch
                buf.append(int(ifmap[idx]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

# ---- BIAS packets ----

def build_bias_l0(bias):
    """
    Layer 0 bias: 128ch × 64pos = 8192 words.
    BRAM_n: Ch[n, n+16, ..., n+112] (8 page) × 64pos.
    DDR: OFFSET_L0 + ch × 64 + pos
    """
    buf = _header(512)
    for bram_id in range(16):
        for page in range(8):
            ch = page * 16 + bram_id
            for pos in range(64):
                idx = BIAS_OFFSET_L0 + ch * 64 + pos
                buf.append(int(bias[idx]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

def build_bias_l1(bias):
    """
    Layer 1 bias: 64ch × 128pos = 8192 words.
    BRAM_n: Ch[n, n+16, n+32, n+48] (4 page) × 128pos.
    DDR: OFFSET_L1 + ch × 128 + pos
    """
    buf = _header(512)
    for bram_id in range(16):
        for page in range(4):
            ch = page * 16 + bram_id
            for pos in range(128):
                idx = BIAS_OFFSET_L1 + ch * 128 + pos
                buf.append(int(bias[idx]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

def build_bias_l2(bias):
    """
    Layer 2 bias: 32ch × 256pos = 8192 words.
    BRAM_n: Ch[n, n+16] (2 page) × 256pos.
    DDR: OFFSET_L2 + ch × 256 + pos
    """
    buf = _header(512)
    for bram_id in range(16):
        for page in range(2):
            ch = page * 16 + bram_id
            for pos in range(256):
                idx = BIAS_OFFSET_L2 + ch * 256 + pos
                buf.append(int(bias[idx]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

def build_bias_l3(bias):
    """
    Layer 3 bias: 16ch × 512pos = 8192 words.
    BRAM_n = channel n, isi 512 pos.
    DDR: OFFSET_L3 + bram_id × 512 + pos
    """
    buf = _header(512)
    for bram_id in range(16):
        for pos in range(512):
            idx = BIAS_OFFSET_L3 + bram_id * 512 + pos
            buf.append(int(bias[idx]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

print("Packet builder functions defined")

In [ ]:
# ============================================================
# CELL 5: Helper — DMA Transfer Functions
# ============================================================

def dma_send(dma, packet_np):
    """
    Kirim packet via AXI DMA send channel.
    packet_np: numpy array uint32.
    """
    buf = allocate(shape=(len(packet_np),), dtype=np.uint32)
    np.copyto(buf, packet_np)
    dma.sendchannel.transfer(buf)
    dma.sendchannel.wait()
    buf.freebuffer()


def send_parallel(dma_w, pkt_weight, dma_i, pkt_ifmap, dma_b, pkt_bias):
    """
    Kirim weight, ifmap, dan bias secara PARALEL (mimik fork–join di testbench).
    Masing-masing berjalan di thread terpisah.
    """
    results = {}  # untuk menyimpan exception jika ada

    def _send(label, dma, pkt):
        try:
            dma_send(dma, pkt)
        except Exception as e:
            results[label] = e

    tw = threading.Thread(target=_send, args=("weight", dma_w, pkt_weight))
    ti = threading.Thread(target=_send, args=("ifmap",  dma_i, pkt_ifmap))
    tb = threading.Thread(target=_send, args=("bias",   dma_b, pkt_bias))

    tw.start(); ti.start(); tb.start()
    tw.join();  ti.join();  tb.join()

    for label, exc in results.items():
        raise RuntimeError(f"DMA send error on '{label}': {exc}")


def recv_notification(dma_m0):
    """
    Terima 1 paket notifikasi dari FPGA via m0_axis.
    Notifikasi = 6 kata, header[0][15:0] == 0xC0DE.
    Return: batch_id (3-bit)
    """
    NOTIF_WORDS = 6
    buf = allocate(shape=(NOTIF_WORDS,), dtype=np.uint32)
    dma_m0.recvchannel.transfer(buf)
    dma_m0.recvchannel.wait()
    magic    = int(buf[0]) & 0xFFFF
    batch_id = int(buf[2]) & 0x7
    buf.freebuffer()
    assert magic == 0xC0DE, f"Notif magic error: got 0x{magic:04X}, expected 0xC0DE"
    return batch_id


def recv_output(dma_m0, dma_m1):
    """
    Terima paket output dari FPGA (m0 + m1 secara paralel).
    m0: 6-word header (0xDA7A) + 4096 data words
    m1: 4096 data words (tanpa header)
    Return: (m0_data_4096, m1_data_4096) sebagai numpy int32
    """
    HDR   = 6
    WORDS = 4096

    buf_m0 = allocate(shape=(HDR + WORDS,), dtype=np.uint32)
    buf_m1 = allocate(shape=(WORDS,),       dtype=np.uint32)

    # Mulai kedua receive secara bersamaan (FPGA akan tunggu tready)
    dma_m0.recvchannel.transfer(buf_m0)
    dma_m1.recvchannel.transfer(buf_m1)
    dma_m0.recvchannel.wait()
    dma_m1.recvchannel.wait()

    magic    = int(buf_m0[0]) & 0xFFFF
    layer_id = int(buf_m0[2]) & 0x3
    assert magic == 0xDA7A, f"Output magic error: got 0x{magic:04X}, expected 0xDA7A"

    # Ambil data saja (skip header)
    raw_m0 = np.array(buf_m0[HDR:], dtype=np.uint32)
    raw_m1 = np.array(buf_m1,       dtype=np.uint32)

    buf_m0.freebuffer()
    buf_m1.freebuffer()

    # Sign-extend 24-bit ke int32
    def sx24(arr):
        a = arr.astype(np.int32) & 0xFFFFFF
        return np.where(a >= 0x800000, a - 0x1000000, a)

    return sx24(raw_m0), sx24(raw_m1), layer_id


print("DMA helper functions defined")

In [ ]:
# ============================================================
# CELL 6: Helper — Output Decoder & Save to File
# ============================================================
# Mapping BRAM → Channel:
#   bram_id = ch % 16
#   page    = ch // 16
#   BRAM 0-7  → m0_data[bram_id * 512 + bram_addr]
#   BRAM 8-15 → m1_data[(bram_id-8) * 512 + bram_addr]

def decode_output(m0_data, m1_data, num_ch, num_pos):
    """
    Susun ulang flat buffer m0/m1 menjadi array [ch, pos].
    m0_data, m1_data: numpy int32, panjang 4096 masing-masing.
    num_ch  : jumlah channel output layer ini
    num_pos : jumlah posisi output layer ini
    Setiap BRAM memiliki 512 entry.
    """
    output = np.zeros((num_ch, num_pos), dtype=np.int32)
    for ch in range(num_ch):
        bram_id  = ch % 16
        page     = ch // 16
        for pos in range(num_pos):
            bram_addr = page * num_pos + pos
            if bram_id < 8:
                output[ch, pos] = m0_data[bram_id * 512 + bram_addr]
            else:
                output[ch, pos] = m1_data[(bram_id - 8) * 512 + bram_addr]
    return output


def save_output_perchannel(output, filename, layer_name):
    """
    Simpan output [num_ch, num_pos] ke file teks per-channel,
    format sama dengan testbench (satu nilai per baris).
    """
    num_ch, num_pos = output.shape
    with open(filename, 'w') as f:
        f.write("=================================================\n")
        f.write(f"{layer_name} OUTPUT - PER CHANNEL DUMP\n")
        f.write(f"Total: {num_ch} Channels x {num_pos} Positions\n")
        f.write("=================================================\n")
        for ch in range(num_ch):
            f.write(f"\n=== CHANNEL {ch} ===\n")
            for pos in range(num_pos):
                f.write(f"{int(output[ch, pos])}\n")
        f.write("\n=================================================\n")
    print(f"  Saved: {filename}")


def decode_l3_output(m0_data, m1_data):
    """
    Layer 3 khusus: 16ch × 512pos.
    m0 = ch 0-7, m1 = ch 8-15.
    Flat layout: m0_buf[ch * 512 + pos], m1_buf[(ch-8)*512 + pos]
    """
    output = np.zeros((16, 512), dtype=np.int32)
    for ch in range(8):
        output[ch, :]     = m0_data[ch * 512 : (ch + 1) * 512]
    for ch in range(8):
        output[ch + 8, :] = m1_data[ch * 512 : (ch + 1) * 512]
    return output


print("Output decoder functions defined")

---
## Pre-build Semua Paket
Build semua packet sebelum kirim ke FPGA agar tidak ada delay saat transfer.

In [ ]:
# ============================================================
# CELL 7: Pre-build semua paket
# ============================================================
import time

t0 = time.time()
print("Building packets...")

# Layer 0 — 8 batches weight
print("  Layer 0 weights (8 batches)...", end=' ', flush=True)
pkt_w_l0 = [build_weight_l0(b, weight_data) for b in range(8)]
print(f"{sum(len(p) for p in pkt_w_l0):,} words total")

print("  Layer 0 ifmap...", end=' ', flush=True)
pkt_i_l0 = build_ifmap_l0(ifmap_l0)
print(f"{len(pkt_i_l0):,} words")

print("  Layer 0 bias...", end=' ', flush=True)
pkt_b_l0 = build_bias_l0(bias_data)
print(f"{len(pkt_b_l0):,} words")

# Layer 1 — 4 batches weight
print("  Layer 1 weights (4 batches)...", end=' ', flush=True)
pkt_w_l1 = [build_weight_l1(b, weight_data) for b in range(4)]
print(f"{sum(len(p) for p in pkt_w_l1):,} words total")

print("  Layer 1 ifmap...", end=' ', flush=True)
pkt_i_l1 = build_ifmap_l1(ifmap_l1)
print(f"{len(pkt_i_l1):,} words")

print("  Layer 1 bias...", end=' ', flush=True)
pkt_b_l1 = build_bias_l1(bias_data)
print(f"{len(pkt_b_l1):,} words")

# Layer 2 — 1 batch
print("  Layer 2 weight...", end=' ', flush=True)
pkt_w_l2 = build_weight_l2(weight_data)
print(f"{len(pkt_w_l2):,} words")

print("  Layer 2 ifmap...", end=' ', flush=True)
pkt_i_l2 = build_ifmap_l2(ifmap_l2)
print(f"{len(pkt_i_l2):,} words")

print("  Layer 2 bias...", end=' ', flush=True)
pkt_b_l2 = build_bias_l2(bias_data)
print(f"{len(pkt_b_l2):,} words")

# Layer 3 — 1 batch
print("  Layer 3 weight...", end=' ', flush=True)
pkt_w_l3 = build_weight_l3(weight_data)
print(f"{len(pkt_w_l3):,} words")

print("  Layer 3 ifmap...", end=' ', flush=True)
pkt_i_l3 = build_ifmap_l3(ifmap_l3)
print(f"{len(pkt_i_l3):,} words")

print("  Layer 3 bias...", end=' ', flush=True)
pkt_b_l3 = build_bias_l3(bias_data)
print(f"{len(pkt_b_l3):,} words")

print(f"\nBuild selesai dalam {time.time()-t0:.1f} detik")

---
## Layer 0 — 256IC × 128OC, 32pos→64pos, 8 batches

In [ ]:
# ============================================================
# CELL 8: Layer 0 Execution
# ============================================================
print("===== LAYER 0 START =====")
t_l0 = time.time()

# Mulai receive notifikasi batch 0 SEBELUM kirim paket
# (FPGA akan kirim notif setelah selesai proses batch 0)

# --- Batch 0: kirim bias + ifmap + weight[0] paralel ---
print("  Sending batch 0 (bias + ifmap + weight[0] parallel)...")
send_parallel(dma_weight, pkt_w_l0[0],
              dma_ifmap,  pkt_i_l0,
              dma_bias,   pkt_b_l0)

print("  Waiting notification batch 0...")
bid = recv_notification(dma_weight)
assert bid == 0, f"Batch mismatch: got {bid}"
print(f"    -> Notif batch {bid} OK")

# --- Batch 1–7: hanya weight ---
for b in range(1, 8):
    print(f"  Sending weight batch {b}...")
    dma_send(dma_weight, pkt_w_l0[b])
    print(f"  Waiting notification batch {b}...")
    bid = recv_notification(dma_weight)
    assert bid == b, f"Batch mismatch: got {bid}, expected {b}"
    print(f"    -> Notif batch {bid} OK")

# --- Terima output setelah semua batch selesai ---
print("  Waiting output...")
m0_raw, m1_raw, lyr = recv_output(dma_weight, dma_ifmap)
print(f"  Output received (layer_id={lyr})")

# Decode: Layer 0 → 128ch × 64pos
out_l0 = decode_output(m0_raw, m1_raw, num_ch=128, num_pos=64)
save_output_perchannel(out_l0, "layer0_output_perchannel.txt", "LAYER 0 (D1)")

print(f"===== LAYER 0 DONE ({time.time()-t_l0:.2f}s) =====")

---
## Layer 1 — 256IC × 64OC, 64pos→128pos, 4 batches

In [ ]:
# ============================================================
# CELL 9: Layer 1 Execution
# ============================================================
print("===== LAYER 1 START =====")
t_l1 = time.time()

# --- Batch 0: bias + ifmap + weight[0] paralel ---
print("  Sending batch 0 (bias + ifmap + weight[0] parallel)...")
send_parallel(dma_weight, pkt_w_l1[0],
              dma_ifmap,  pkt_i_l1,
              dma_bias,   pkt_b_l1)

print("  Waiting notification batch 0...")
bid = recv_notification(dma_weight)
assert bid == 0
print(f"    -> Notif batch {bid} OK")

# --- Batch 1–3 ---
for b in range(1, 4):
    print(f"  Sending weight batch {b}...")
    dma_send(dma_weight, pkt_w_l1[b])
    print(f"  Waiting notification batch {b}...")
    bid = recv_notification(dma_weight)
    assert bid == b
    print(f"    -> Notif batch {bid} OK")

# --- Output ---
print("  Waiting output...")
m0_raw, m1_raw, lyr = recv_output(dma_weight, dma_ifmap)
print(f"  Output received (layer_id={lyr})")

# Decode: Layer 1 → 64ch × 128pos
out_l1 = decode_output(m0_raw, m1_raw, num_ch=64, num_pos=128)
save_output_perchannel(out_l1, "layer1_output_perchannel.txt", "LAYER 1 (D2)")

print(f"===== LAYER 1 DONE ({time.time()-t_l1:.2f}s) =====")

---
## Layer 2 — 128IC × 32OC, 128pos→256pos, 1 batch

In [ ]:
# ============================================================
# CELL 10: Layer 2 Execution
# ============================================================
print("===== LAYER 2 START =====")
t_l2 = time.time()

# 1 batch saja: bias + ifmap + weight paralel
print("  Sending batch 0 (bias + ifmap + weight parallel)...")
send_parallel(dma_weight, pkt_w_l2,
              dma_ifmap,  pkt_i_l2,
              dma_bias,   pkt_b_l2)

print("  Waiting notification batch 0...")
bid = recv_notification(dma_weight)
assert bid == 0
print(f"    -> Notif batch {bid} OK")

print("  Waiting output...")
m0_raw, m1_raw, lyr = recv_output(dma_weight, dma_ifmap)
print(f"  Output received (layer_id={lyr})")

# Decode: Layer 2 → 32ch × 256pos
out_l2 = decode_output(m0_raw, m1_raw, num_ch=32, num_pos=256)
save_output_perchannel(out_l2, "layer2_output_perchannel.txt", "LAYER 2 (D3)")

print(f"===== LAYER 2 DONE ({time.time()-t_l2:.2f}s) =====")

---
## Layer 3 — 64IC × 16OC, 256pos→512pos, 1 batch

In [ ]:
# ============================================================
# CELL 11: Layer 3 Execution
# ============================================================
print("===== LAYER 3 START =====")
t_l3 = time.time()

# 1 batch saja: bias + ifmap + weight paralel
print("  Sending batch 0 (bias + ifmap + weight parallel)...")
send_parallel(dma_weight, pkt_w_l3,
              dma_ifmap,  pkt_i_l3,
              dma_bias,   pkt_b_l3)

print("  Waiting notification batch 0...")
bid = recv_notification(dma_weight)
assert bid == 0
print(f"    -> Notif batch {bid} OK")

print("  Waiting output...")
m0_raw, m1_raw, lyr = recv_output(dma_weight, dma_ifmap)
print(f"  Output received (layer_id={lyr})")

# Layer 3 khusus: layout flat berbeda
out_l3 = decode_l3_output(m0_raw, m1_raw)

# Simpan per-channel
with open("layer3_output_perchannel.txt", 'w') as f:
    f.write("=================================================\n")
    f.write("LAYER 3 (D5) OUTPUT - PER CHANNEL DUMP\n")
    f.write("Format: Channel -> 512 Positions\n")
    f.write("Total: 16 Channels x 512 Positions = 8192 values\n")
    f.write("=================================================\n")
    for ch in range(16):
        f.write(f"\n=== CHANNEL {ch} ===\n")
        for pos in range(512):
            f.write(f"{int(out_l3[ch, pos])}\n")
    f.write("\n=================================================\n")
print("  Saved: layer3_output_perchannel.txt")

print(f"===== LAYER 3 DONE ({time.time()-t_l3:.2f}s) =====")

In [ ]:
# ============================================================
# CELL 12: Ringkasan & Preview Output
# ============================================================
print("\n========================================")
print(" INFERENCE COMPLETE")
print("========================================")
print(f" Layer 0: {out_l0.shape}  → layer0_output_perchannel.txt")
print(f" Layer 1: {out_l1.shape}  → layer1_output_perchannel.txt")
print(f" Layer 2: {out_l2.shape}  → layer2_output_perchannel.txt")
print(f" Layer 3: {out_l3.shape}  → layer3_output_perchannel.txt")
print("========================================")

# Preview: 5 nilai pertama tiap layer
print("\nPreview Layer 0 Ch0:", out_l0[0, :5])
print("Preview Layer 1 Ch0:", out_l1[0, :5])
print("Preview Layer 2 Ch0:", out_l2[0, :5])
print("Preview Layer 3 Ch0:", out_l3[0, :5])

---
## Catatan Konfigurasi PYNQ

### Nama DMA dalam Overlay
Sesuaikan nama `axi_dma_0/1/2` di Cell 2 dengan nama IP di Vivado block design.
Cek dengan:
```python
print(overlay.ip_dict.keys())
```

### Ukuran Buffer DMA
| Paket | Jumlah kata (uint32) | Bytes |
|-------|---------------------|-------|
| Weight L0/batch | 16390 | ~64 KB |
| Weight L1/batch | 16390 | ~64 KB |
| Weight L2       | 16390 | ~64 KB |
| Weight L3       | 4102  | ~16 KB |
| IFMAP L0        | 8198  | ~32 KB |
| IFMAP L1/L2/L3  | 16390 | ~64 KB |
| Bias L0/1/2/3   | 8198  | ~32 KB |
| Output m0       | 4102  | ~16 KB |
| Output m1       | 4096  | ~16 KB |
| Notifikasi      | 6     | 24 B   |

### Pengaturan `tlast`
DMA `tlast` harus diaktifkan agar tiap paket berakhir dengan benar.
Di Vivado AXI DMA settings: **Allow Unaligned Transfers = No**, **Enable Scatter Gather = No** (simple mode).

### Data Width
DUT menggunakan `DW=24` bit. Jika DMA dikonfigurasi 32-bit, bit [31:24] akan diabaikan FPGA (parser sudah mengambil `tdata[23:0]` saja).